In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, accuracy_score



In [9]:
#clean and format sheet
data_path = r"C:\Users\Dalyn\OneDrive - Pace University\Desktop\Capstone\ML\datasets\against_filtered.csv"
import time_conversion
import data_clean
import against_filter

time_conversion.convert_excel_time_column_to_minutes(data_path, "Time", data_path)
data_clean.delete_rows_with_missing_data(data_path, data_path)
against_filter.extract_second_name_from_column(data_path, "against", data_path)

Conversion successful. Results saved to C:\Users\Dalyn\OneDrive - Pace University\Desktop\Capstone\ML\datasets\against_filtered.csv
Rows with missing data deleted. Results saved to C:\Users\Dalyn\OneDrive - Pace University\Desktop\Capstone\ML\datasets\against_filtered.csv
Second names extracted. Results saved to C:\Users\Dalyn\OneDrive - Pace University\Desktop\Capstone\ML\datasets\against_filtered.csv


In [37]:
try:
    df = pd.read_csv(r"C:\Users\Dalyn\OneDrive - Pace University\Desktop\Capstone\ML\datasets\against_filtered.csv")
    print(df.head())
except FileNotFoundError:
    print ("error dataset not found. Please provide the correct file path.")



            Name      Date        Tournament Surface   Rd  Rk  vRk  \
0  Jannik Sinner  2-Oct-24  Shanghai Masters    Hard    F   1    4   
1  Jannik Sinner  2-Oct-24  Shanghai Masters    Hard   SF   1   33   
2  Jannik Sinner  2-Oct-24  Shanghai Masters    Hard   QF   1    5   
3  Jannik Sinner  2-Oct-24  Shanghai Masters    Hard  R16   1   16   
4  Jannik Sinner  2-Oct-24  Shanghai Masters    Hard  R32   1   37   

                                           against           Score   TP  Aces  \
0           ['(1)Sinner', '(4)NovakDjokovic[SRB]']      7-6(4) 6-3  125     8   
1            ['(1)Sinner', '(30)TomasMachac[CZE]']         6-4 7-5  138    10   
2      ['(1)Sinner', '(5)DaniilMe', 'e', 'v[RUS]']         6-1 6-4  108     9   
3             ['(1)Sinner', '(14)BenShelton[USA]']      6-4 7-6(1)  124     7   
4  ['(1)Sinner', '(31)TomasMartinEtcheverry[ARG]']  6-7(3) 6-4 6-2  186    12   

   DFs  SP  1SP  2SP  vA  Time           Against_name  Time_minutes  
0    0  67   41   26  

In [12]:
df = df.dropna(subset=['Score'])
df = df.dropna(subset=['Name', 'Against_name', 'Rd', 'Rk', 'vRk', 'Tournament', 'Surface'])

df['Result'] = df.apply(lambda row: 1 if row['Name'] in row['Score'] else 0, axis=1)


In [13]:
label_encoders = {}
categorical_cols = ['Name', 'Against_name', 'Rd', 'Tournament', 'Surface']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [14]:
features = ['Name', 'Against_name', 'Rd', 'Rk', 'vRk', 'Tournament', 'Surface']
target_numerical = ['TP', 'Aces', 'DFs', 'SP', '1SP', '2SP', 'vA', 'Time_minutes']
target_classification = ['Result']

In [15]:
numerical_cols = ['Rk', 'vRk']
imputer = SimpleImputer(strategy='mean')
df[numerical_cols] = imputer.fit_transform(df[numerical_cols])

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [16]:
X = df[features]
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
models_reg = {}

In [17]:
for target in target_numerical:
    y = df[target]
    y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    models_reg[target] = model
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"MSE for {target}: {mse}")

MSE for TP: 1989.3874578932991
MSE for Aces: 12.92158232357273
MSE for DFs: 4.9227530213331905
MSE for SP: 538.9242893402968
MSE for 1SP: 237.86911994865082
MSE for 2SP: 111.6254506850208
MSE for vA: 16.004480338773607
MSE for Time_minutes: 1226.573606826237


In [23]:
model_class = RandomForestClassifier(random_state=42)
y_class = df['Result']
y_train_class, y_test_class = train_test_split(y_class, test_size=0.2, random_state=42)
model_class.fit(X_train, y_train_class)
y_pred_class = model_class.predict(X_test)
accuracy = accuracy_score(y_test_class, y_pred_class)
print(f"Accuracy for Result: {accuracy}")

Accuracy for Result: 1.0


In [30]:
player_name = "Novak Djokovic"
opponent_name = "CarlosAlcaraz"
round_val = "F"
player_rank = 1
opponent_rank = 2
tournament = "Wimbledon"
surface = "Grass"



In [31]:
try:
    input_data = pd.DataFrame({
        'Name': [label_encoders['Name'].transform([player_name])[0]],
        'Against_name': [label_encoders['Against_name'].transform([opponent_name])[0]],
        'Rd': [label_encoders['Rd'].transform([round_val])[0]],
        'Rk': [player_rank],
        'vRk': [opponent_rank],
        'Tournament': [label_encoders['Tournament'].transform([tournament])[0]],
        'Surface': [label_encoders['Surface'].transform([surface])[0]]
    })
except ValueError as e:
    if "y contains previously unseen labels" in str(e):
        print ("error One or more of the provided player, opponent, tournament, round, or surface values were not found in the training data.")
    else:
        print ("error"+ str(e))

In [32]:
input_data[numerical_cols] = imputer.transform(input_data[numerical_cols])
input_data[numerical_cols] = scaler.transform(input_data[numerical_cols])

In [33]:
predictions_reg = {}
for target, model in models_reg.items():
    predictions_reg[target] = model.predict(input_data)[0]

In [36]:
prediction_class = model_class.predict(input_data)[0]
result = "Win" if prediction_class == 1 else "Loss"

predictions_reg['Result'] = result
print(result)


Loss
